# 🚦 DQN Traffic Light Control — Quick Start

Notebook này demo nhanh toàn bộ pipeline của dự án:
- Tạo kịch bản ngã tư Hà Nội ảo (SUMO)
- Test môi trường mô phỏng
- Khởi tạo DQN agent
- Load model đã train và chạy thử

> **Yêu cầu:** Đã cài SUMO và set `SUMO_HOME`. Xem README.md để biết cách cài đặt.


In [5]:
# Cài dependencies nếu chưa có
# !pip install -r requirements.txt

# Set SUMO_HOME cho Windows (chỉnh lại đường dẫn nếu cài khác)
import os
sumo_path = r"C:\Program Files (x86)\Eclipse\Sumo"
if os.path.isdir(sumo_path):
    os.environ["SUMO_HOME"] = sumo_path
    print(f"✓ SUMO_HOME = {sumo_path}")
elif "SUMO_HOME" in os.environ:
    print(f"✓ SUMO_HOME đã được set: {os.environ['SUMO_HOME']}")
else:
    print("⚠️  SUMO_HOME chưa được set — hãy chỉnh đường dẫn ở dòng trên")


✓ SUMO_HOME = C:\Program Files (x86)\Eclipse\Sumo


## Bước 1 — Tạo kịch bản SUMO (ngã tư Hà Nội mẫu)

Script dưới đây tạo ra các file XML mô tả ngã tư ảo:
- `nodes.nod.xml` — 5 điểm nút (trung tâm + 4 hướng)
- `edges.edg.xml` — 8 đường (4 vào, 4 ra), mỗi đường 2 làn
- `routes.rou.xml` — Loại xe VN (60% xe máy, 30% ô tô, 10% buýt/tải) + luồng giao thông
- `config.sumocfg` — Cấu hình chạy SUMO

> Nếu đã có folder `data/scenarios/hn_sample/` rồi thì bỏ qua bước này.


In [6]:
from src.utils.generate_scenario import generate_simple_intersection

# Tạo kịch bản ngã tư Hà Nội với phân bố phương tiện Việt Nam
generate_simple_intersection("data/scenarios/hn_sample")
print("✓ Đã tạo xong các file XML kịch bản")


✓ Created scenario files in data/scenarios/hn_sample/
  Next: run 'netconvert --node-files=data/scenarios/hn_sample/nodes.nod.xml --edge-files=data/scenarios/hn_sample/edges.edg.xml --output-file=data/scenarios/hn_sample/intersection.net.xml'
✓ Đã tạo xong các file XML kịch bản


## Bước 2 — Build mạng SUMO (netconvert)

Chạy lệnh sau trong **PowerShell** (chỉ cần chạy 1 lần):

```powershell
netconvert `
  --node-files=data/scenarios/hn_sample/nodes.nod.xml `
  --edge-files=data/scenarios/hn_sample/edges.edg.xml `
  --output-file=data/scenarios/hn_sample/intersection.net.xml `
  --no-turnarounds true
```

> File `intersection.net.xml` là file mạng biên dịch — SUMO đọc file này để biết cấu trúc đường.  
> Nếu file này đã tồn tại trong `data/scenarios/hn_sample/` rồi thì **bỏ qua bước này**.


## Bước 3 — Kiểm tra môi trường mô phỏng

Tạo môi trường SUMO và chạy thử 20 bước với **action ngẫu nhiên** (không có AI).  
Mục đích: xác nhận SUMO hoạt động đúng và state/reward trả về hợp lệ.


In [7]:
from src.env.sumo_env import SumoMDPEnv, EnvConfig, VNWeights
import numpy as np

# Cấu hình môi trường — đồng bộ với config.yaml
cfg = EnvConfig(
    sumocfg_path="data/scenarios/hn_sample/config.sumocfg",
    tls_id="c",                    # ID đèn giao thông trong intersection.net.xml
    phases=[0, 1, 2, 3],          # 4 pha: NS-thẳng, NS-trái, ĐT-thẳng, ĐT-trái
    action_duration=5,             # Giữ mỗi pha 5 giây
    max_steps=200,                 # Chạy 200 bước để test nhanh
    warmup_steps=60,               # 60 bước khởi động trước khi học
    gui=False,                     # Tắt GUI để chạy nhanh hơn
    vn_weights=VNWeights(          # Trọng số PCU chuẩn Việt Nam
        motorcycle=0.5,            # 1 ô tô ≈ 3 xe máy
        car=1.5,
        bus=2.0,
        truck=2.0,
    ),
)

env = SumoMDPEnv(cfg)
state = env.reset()

print(f"✓ Môi trường khởi tạo thành công")
print(f"  State dim  : {env.state_dim}  (8 làn × 4 đặc trưng = 32)")
print(f"  Action dim : {env.action_dim}  (4 pha đèn)")
print(f"  State mẫu  : {state[:8].round(3)}  ...")

# Chạy 20 bước với action ngẫu nhiên
total_r = 0.0
for step in range(20):
    a = np.random.randint(env.action_dim)
    s2, r, done, info = env.step(a)
    total_r += r
    if done:
        break

env.close()
print(f"\n✓ Chạy xong 20 bước ngẫu nhiên")
print(f"  Tổng reward : {total_r:.2f}  (âm = có tắc đường, gần 0 = thông thoáng)")
print(f"  Queue TB    : {info['queue_length']:.1f} xe")
print(f"  Tốc độ TB   : {info['avg_speed']:.2f} m/s")


✓ Môi trường khởi tạo thành công
  State dim  : 32  (8 làn × 4 đặc trưng = 32)
  Action dim : 4  (4 pha đèn)
  State mẫu  : [ 1.     9.178  0.     2.     0.    13.9    0.     0.   ]  ...

✓ Chạy xong 20 bước ngẫu nhiên
  Tổng reward : -223.42  (âm = có tắc đường, gần 0 = thông thoáng)
  Queue TB    : 22.0 xe
  Tốc độ TB   : 5.96 m/s


## Bước 4 — Khởi tạo DQN Agent

Tạo agent với cấu hình khớp `config.yaml`. Cell này **không train**, chỉ khởi tạo.  
Để train đầy đủ (200k bước) → chạy `python scripts/train.py` trong terminal.


In [8]:
import torch
from src.dqn.agent import DQNAgent, AgentConfig
from src.dqn.replay_buffer import ReplayBuffer
from src.utils import LinearEpsilon

# Cấu hình agent — đồng bộ với config.yaml / scripts/train.py
agent_cfg = AgentConfig(
    state_dim=32,           # 8 làn × 4 đặc trưng
    action_dim=4,           # 4 pha đèn
    gamma=0.99,             # Hệ số chiết khấu dài hạn
    lr=5e-4,                # Learning rate Adam
    batch_size=64,
    tau=1.0,                # Hard update (copy toàn bộ weights)
    target_update_interval=2000,
    double_dqn=True,        # Dùng Double DQN để tránh overestimate
    grad_clip=10.0,
)

agent = DQNAgent(agent_cfg)
buffer = ReplayBuffer(100_000, (32,))
eps_sched = LinearEpsilon(start=1.0, end=0.05, steps=100_000)

print("✓ Agent khởi tạo thành công")
print(f"  Device   : {agent.device}")
print(f"  Network  : Dueling DQN (state=32 → hidden=128×2 → action=4)")
print(f"  Double DQN: {agent_cfg.double_dqn}")
print(f"\n→ Để train: chạy 'python scripts/train.py' trong terminal (~30-60 phút)")
print(f"→ Model sẽ lưu tại: outputs/dqn_vn_tls.pt")


✓ Agent khởi tạo thành công
  Device   : cuda
  Network  : Dueling DQN (state=32 → hidden=128×2 → action=4)
  Double DQN: True

→ Để train: chạy 'python scripts/train.py' trong terminal (~30-60 phút)
→ Model sẽ lưu tại: outputs/dqn_vn_tls.pt


## Bước 5 — Load model đã train và chạy thử (greedy)

Sau khi đã có `outputs/dqn_vn_tls.pt`, chạy cell dưới để xem agent hoạt động:


In [9]:
from pathlib import Path

model_path = Path("outputs/dqn_vn_tls.pt")

if not model_path.exists():
    print("⚠️  Chưa có model. Chạy 'python scripts/train.py' trước.")
else:
    # Load trọng số vào agent
    agent.q.load_state_dict(torch.load(model_path, map_location="cpu"))
    agent.q.eval()
    print(f"✓ Load model thành công: {model_path}")

    # Chạy môi trường với DQN greedy (eps=0 = không random)
    env2 = SumoMDPEnv(cfg)
    state = env2.reset()

    total_r = 0.0
    queue_list = []
    wait_list  = []

    for step in range(100):
        action = agent.act(state, eps=0.0)          # Greedy — không random
        state, r, done, info = env2.step(action)
        total_r += r
        queue_list.append(info["queue_length"])
        wait_list.append(info["avg_wait"])
        if done:
            break

    env2.close()

    print(f"\n📊 Kết quả DQN (100 bước greedy):")
    print(f"  Tổng reward       : {total_r:.2f}")
    print(f"  Queue TB          : {sum(queue_list)/len(queue_list):.2f} xe")
    print(f"  Thời gian chờ TB  : {sum(wait_list)/len(wait_list):.2f} s")
    print(f"\n→ Xem trực quan: python scripts/gui.py dqn")
    print(f"→ So sánh đầy đủ: python scripts/parallel_comparison.py")


✓ Load model thành công: outputs\dqn_vn_tls.pt

📊 Kết quả DQN (100 bước greedy):
  Tổng reward       : -1084.82
  Queue TB          : 22.62 xe
  Thời gian chờ TB  : 33.72 s

→ Xem trực quan: python scripts/gui.py dqn
→ So sánh đầy đủ: python scripts/parallel_comparison.py
